# 🔗 MinIO S3 Data Explorer

**Mục đích**: Kết nối MinIO, tải parquet từ data lake và hiển thị nội dung

Notebook này cung cấp một môi trường interactive để:
- Kết nối với MinIO S3
- Liệt kê buckets và files
- Tải dữ liệu Parquet
- Hiển thị dữ liệu chi tiết

In [38]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

# Tất cả packages cần thiết cho notebook
packages = [
    'minio',
    'pandas',
    'pyarrow',
    'confluent-kafka',
    'fastavro',
    'cachetools',
    'httpx',
    'attrs',
    'authlib',
    'requests'
]

print("📦 Cài đặt thư viện...\n")
failed_packages = []

for package in packages:
    try:
        # Thử import package (handle package name khác import name)
        if package == 'confluent-kafka':
            __import__('confluent_kafka')
        elif package == 'pyarrow':
            __import__('pyarrow')
        elif package == 'httpx':
            __import__('httpx')
        elif package == 'cachetools':
            __import__('cachetools')
        else:
            __import__(package)
        print(f"✅ {package:<25} - đã có sẵn")
    except ImportError as e:
        print(f"⬇️  Đang cài {package:<25}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"✅ {package:<25} - cài xong")
        except Exception as err:
            print(f"❌ {package:<25} - LỖI: {err}")
            failed_packages.append(package)

if failed_packages:
    print(f"\n⚠️  Các package cài đặt thất bại: {failed_packages}")
else:
    print("\n✨ Tất cả thư viện sẵn sàng!")

📦 Cài đặt thư viện...

✅ minio                     - đã có sẵn
✅ pandas                    - đã có sẵn
✅ pyarrow                   - đã có sẵn
✅ confluent-kafka           - đã có sẵn
✅ fastavro                  - đã có sẵn
✅ cachetools                - đã có sẵn
✅ httpx                     - đã có sẵn
✅ attrs                     - đã có sẵn
✅ authlib                   - đã có sẵn
⬇️  Đang cài requests                 ...
✅ requests                  - cài xong

✨ Tất cả thư viện sẵn sàng!


In [1]:
# Import thư viện
import os
from minio import Minio
from minio.error import S3Error
import pandas as pd
import io

# Cấu hình kết nối MinIO
MINIO_CONFIG = {
    'endpoint': os.getenv('MINIO_ENDPOINT', 'localhost:9000'),
    'access_key': os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    'secret_key': os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    'region': os.getenv('MINIO_REGION', 'ap-southeast-1'),
    'use_ssl': False
}

print(f"🔧 Kết nối MinIO tại: {MINIO_CONFIG['endpoint']}")

try:
    # Khởi tạo client MinIO
    client = Minio(
        endpoint=MINIO_CONFIG['endpoint'],
        access_key=MINIO_CONFIG['access_key'],
        secret_key=MINIO_CONFIG['secret_key'],
        region=MINIO_CONFIG['region'],
        secure=MINIO_CONFIG['use_ssl']
    )
    
    print("✅ Kết nối MinIO thành công!")
    
except Exception as err:
    print(f"❌ Lỗi kết nối: {err}")
    print(f"💡 Đảm bảo MinIO đang chạy tại {MINIO_CONFIG['endpoint']}")

🔧 Kết nối MinIO tại: localhost:9000
✅ Kết nối MinIO thành công!


In [4]:
# Liệt kê tất cả buckets
print("📦 Danh sách Buckets:\n")

try:
    buckets = list(client.list_buckets())
    
    if buckets:
        for i, bucket in enumerate(buckets, 1):
            print(f"{i}. {bucket.name}")
        print(f"\n✅ Tổng cộng: {len(buckets)} bucket")
    else:
        print("⚠️  Không có bucket nào. Tạo 'data-lake' bucket...")
        client.make_bucket('data-lake')
        print("✅ Bucket 'data-lake' đã tạo")
        
except Exception as err:
    print(f"❌ Lỗi: {err}")

📦 Danh sách Buckets:

1. data-lake

✅ Tổng cộng: 1 bucket


In [53]:
# Tìm và tải file parquet
print("🔍 Tìm file Parquet...\n")

bucket_name = 'data-lake'
parquet_files = []

try:
    # Tìm tất cả file .parquet
    for obj in client.list_objects(bucket_name, recursive=True):
        if obj.object_name.endswith('.parquet'):
            parquet_files.append(obj.object_name)
    
    if parquet_files:
        print(f"✅ Tìm thấy {len(parquet_files)} file Parquet:\n")
        for i, file in enumerate(parquet_files[:5], 1):
            print(f"{i}. {file}")
        
        if len(parquet_files) > 5:
            print(f"... và {len(parquet_files) - 5} file khác")
    else:
        print("⚠️  Chưa có file Parquet. Hãy chạy Spark job để tạo dữ liệu.")
        
except S3Error as err:
    print(f"❌ Lỗi: {err}")

🔍 Tìm file Parquet...

✅ Tìm thấy 928 file Parquet:

1. warehouse/bronze/bronze_delivery_status/data/event_time_day=2026-04-06/00172-100737-cb89365e-52d0-4bdc-b080-91ce3493baf1-76-00001.parquet
2. warehouse/bronze/bronze_delivery_status/data/event_time_day=2026-04-06/00172-102045-cb89365e-52d0-4bdc-b080-91ce3493baf1-77-00001.parquet
3. warehouse/bronze/bronze_delivery_status/data/event_time_day=2026-04-06/00172-103354-cb89365e-52d0-4bdc-b080-91ce3493baf1-78-00001.parquet
4. warehouse/bronze/bronze_delivery_status/data/event_time_day=2026-04-06/00172-104660-cb89365e-52d0-4bdc-b080-91ce3493baf1-79-00001.parquet
5. warehouse/bronze/bronze_delivery_status/data/event_time_day=2026-04-06/00172-10482-cb89365e-52d0-4bdc-b080-91ce3493baf1-7-00001.parquet
... và 923 file khác


In [54]:
# 🎯 UNIVERSAL TOPIC DECODER & VIEWER

print("\n" + "="*100)
print("🎯 UNIVERSAL TOPIC DECODER - Decode Any Topic")
print("="*100 + "\n")

import fastavro
from io import BytesIO
import requests
import json

SCHEMA_REGISTRY_URL = "http://localhost:8081"
schema_cache = {}

def get_schema_from_registry(schema_id: int, registry_url: str = SCHEMA_REGISTRY_URL):
    """Fetch Avro schema from Schema Registry by schema ID"""
    if schema_id in schema_cache:
        return schema_cache[schema_id]
    
    try:
        url = f"{registry_url}/schemas/ids/{schema_id}"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            schema = json.loads(data.get('schema', '{}'))
            schema_cache[schema_id] = schema
            return schema
    except:
        pass
    
    return None

def deserialize_kafka_avro(raw_bytes):
    """Deserialize Kafka Avro: [0x00][schema_id:4bytes][avro_payload]"""
    try:
        if not isinstance(raw_bytes, (bytes, bytearray)) or len(raw_bytes) < 6:
            return None
        
        if raw_bytes[0:1] != b'\x00':
            return None
        
        schema_id = int.from_bytes(raw_bytes[1:5], 'big')
        payload = raw_bytes[5:]
        
        schema = get_schema_from_registry(schema_id)
        
        if schema:
            bio = BytesIO(payload)
            try:
                record = fastavro.schemaless_reader(bio, schema)
                return record
            except:
                pass
        
        # Fallback for orders
        fallback_schema = {
            "type": "record",
            "name": "Order",
            "fields": [
                {"name": "order_id", "type": "string"},
                {"name": "user_id", "type": "string"},
                {"name": "product_id", "type": "string"},
                {"name": "amount", "type": "double"},
                {"name": "currency", "type": "string"},
                {"name": "ts", "type": "long"}
            ]
        }
        
        bio = BytesIO(payload)
        try:
            record = fastavro.schemaless_reader(bio, fallback_schema)
            return record
        except:
            pass
        
    except:
        pass
    
    return None

def view_topic(topic_name: str, limit: int = 10, show_raw_bytes: bool = False):
    """
    Decode and display any topic with full details
    
    Args:
        topic_name: 'orders', 'payments', 'shipments', 'delivery', 'users', 'products'
        limit: Number of records to show
        show_raw_bytes: Show raw Avro bytes (for debugging)
    """
    
    BRONZE_TABLES = {
        'orders': 'bronze_orders',
        'payments': 'bronze_payments',
        'shipments': 'bronze_shipments',
        'delivery': 'bronze_delivery_status',
        'users': 'bronze_users',
        'products': 'bronze_products'
    }
    
    if topic_name not in BRONZE_TABLES:
        print(f"❌ Topic '{topic_name}' not found!")
        print(f"   Available: {', '.join(BRONZE_TABLES.keys())}")
        return None
    
    table_name = BRONZE_TABLES[topic_name]
    search_terms = [table_name, topic_name]
    
    # Find parquet file
    matching_files = [f for f in parquet_files if any(term in f.lower() for term in search_terms)]
    
    if not matching_files:
        print(f"❌ No files for topic '{topic_name}'")
        return None
    
    matching_files.sort()
    file_to_load = matching_files[-1]
    
    try:
        response = client.get_object(bucket_name, file_to_load)
        data = response.read()
        df = pd.read_parquet(io.BytesIO(data))
        
        # Decode raw_value
        if 'raw_value' in df.columns:
            decoded_records = []
            for idx, row in df.iterrows():
                raw_bytes = row['raw_value']
                decoded = deserialize_kafka_avro(raw_bytes)
                
                record = {
                    'event_source': row.get('event_source', '?'),
                    'event_time': row.get('event_time', '?'),
                }
                
                if isinstance(decoded, dict):
                    record.update(decoded)
                
                decoded_records.append(record)
            
            df_decoded = pd.DataFrame(decoded_records)
        else:
            df_decoded = df.copy()
        
        # Print output
        print(f"\n{'='*100}")
        print(f"📊 TOPIC: {topic_name.upper()} | File: {file_to_load.split('/')[-1]}")
        print(f"{'='*100}")
        print(f"Total records: {len(df_decoded):,} | Showing: {min(limit, len(df_decoded))}\n")
        
        # Print each record
        for idx in range(min(limit, len(df_decoded))):
            row = df_decoded.iloc[idx]
            
            print(f"{'─'*100}")
            print(f"Record #{idx + 1}:")
            print(f"{'─'*100}")
            
            for col in df_decoded.columns:
                val = row[col]
                
                if pd.isna(val):
                    val_str = "[NULL]"
                elif isinstance(val, (int, float)):
                    val_str = f"{val}"
                elif isinstance(val, dict):
                    val_str = str(val)[:80]
                else:
                    val_str = str(val)[:80]
                
                print(f"  {col:<30} = {val_str}")
            
            print()
        
        # Summary stats
        print(f"{'='*100}")
        print(f"📊 Summary Statistics")
        print(f"{'='*100}\n")
        
        numeric_cols = df_decoded.select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            print(f"Numeric columns:")
            print(df_decoded[numeric_cols].describe().T.to_string())
            print()
        
        return df_decoded
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Print usage guide
print("""
╔════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                        🎯 UNIVERSAL TOPIC DECODER - Usage Guide                                   ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝

Available Topics (6 total):
  📤 Streaming Events:
     • 'orders'       → orders.v1          (order_id, user_id, product_id, amount, currency, ts)
     • 'payments'     → payments.v1        (payment_id, order_id, method, status, amount)
     • 'shipments'    → shipments.v1       (shipment_id, order_id, eta_days, carrier)
     • 'delivery'     → delivery-status.v1 (delivery_id, shipment_id, status, reason)
  
  💾 CDC Events (Debezium):
     • 'users'        → demo.public.users    (CDC: user changes with before/after)
     • 'products'     → demo.public.products (CDC: product changes with before/after)

Usage Examples:
  ┌─────────────────────────────────────────────────────────────────────────────────────┐
  │ df = view_topic('orders', limit=5)           # Show 5 orders                        │
  │ df = view_topic('payments', limit=10)        # Show 10 payments                     │
  │ df = view_topic('users', limit=3)            # Show 3 user changes (CDC)            │
  │ df = view_topic('products', limit=15)        # Show 15 product changes              │
  │ df = view_topic('shipments', limit=1, show_raw_bytes=True)  # With raw bytes       │
  └─────────────────────────────────────────────────────────────────────────────────────┘

All data is automatically decoded from Kafka Avro format!
""")

# Demo: Show first 3 orders
print("\n" + "="*100)
print("📌 DEMO: Orders Topic (first 3 records)")
print("="*100)
df_demo = view_topic('orders', limit=3)



🎯 UNIVERSAL TOPIC DECODER - Decode Any Topic


╔════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                        🎯 UNIVERSAL TOPIC DECODER - Usage Guide                                   ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════╝

Available Topics (6 total):
  📤 Streaming Events:
     • 'orders'       → orders.v1          (order_id, user_id, product_id, amount, currency, ts)
     • 'payments'     → payments.v1        (payment_id, order_id, method, status, amount)
     • 'shipments'    → shipments.v1       (shipment_id, order_id, eta_days, carrier)
     • 'delivery'     → delivery-status.v1 (delivery_id, shipment_id, status, reason)

  💾 CDC Events (Debezium):
     • 'users'        → demo.public.users    (CDC: user changes with before/after)
     • 'products'     → demo.public.products (CDC: product changes with before/after)

Usage Examples:
  ┌──────────────────

In [55]:
# ⚡ QUICK TEST ALL TOPICS

print("\n" + "="*100)
print("⚡ QUICK TEST: All 6 Topics")
print("="*100 + "\n")

topics_to_test = {
    'orders': 'Streaming: New Orders',
    'payments': 'Streaming: Payment Events',
    'shipments': 'Streaming: Shipment Events',
    'delivery': 'Streaming: Delivery Status',
    'users': 'CDC: User Changes (before/after)',
    'products': 'CDC: Product Changes (before/after)'
}

print("Uncomment below to view each topic:\n")

for topic in topics_to_test.keys():
    print(f"# view_topic('{topic}', limit=5)         # {topics_to_test[topic]}")

print("\n" + "="*100)
print("Testing all topics (1 record each):")
print("="*100 + "\n")

for topic in topics_to_test.keys():
    print(f"\n▶️  Topic: {topic.upper()}")
    try:
        df = view_topic(topic, limit=1)
        if df is not None:
            print(f"   ✅ Successfully decoded {len(df):,} total records")
        else:
            print(f"   ❌ Failed to load")
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:50]}")

print("\n" + "="*100)
print("✨ All Topics Ready! Use view_topic(topic_name, limit=N) to view any topic")
print("="*100)



⚡ QUICK TEST: All 6 Topics

Uncomment below to view each topic:

# view_topic('orders', limit=5)         # Streaming: New Orders
# view_topic('payments', limit=5)         # Streaming: Payment Events
# view_topic('shipments', limit=5)         # Streaming: Shipment Events
# view_topic('delivery', limit=5)         # Streaming: Delivery Status
# view_topic('users', limit=5)         # CDC: User Changes (before/after)
# view_topic('products', limit=5)         # CDC: Product Changes (before/after)

Testing all topics (1 record each):


▶️  Topic: ORDERS

📊 TOPIC: ORDERS | File: 00172-98992-b6ad8736-41a5-46dc-9ca4-a0727b03afdb-75-00001.parquet
Total records: 1,967 | Showing: 1

────────────────────────────────────────────────────────────────────────────────────────────────────
Record #1:
────────────────────────────────────────────────────────────────────────────────────────────────────
  event_source                   = orders.v1
  event_time                     = 2026-04-06 18:05:57.209000+

In [56]:
# 📌 SIMPLE FUNCTION: Select Topic & Show K Samples

def show_samples(topic: str, k: int = 5):
    """
    Đơn giản: Chọn topic và in ra K mẫu
    
    Args:
        topic: Tên topic ('orders', 'payments', 'shipments', 'delivery', 'users', 'products')
        k: Số mẫu cần hiển thị
    
    Examples:
        show_samples('orders', 3)       # In 3 orders
        show_samples('users', 5)        # In 5 user changes
        show_samples('payments', 10)    # In 10 payments
    """
    
    print(f"\n{'='*100}")
    print(f"📋 Showing {k} samples from '{topic.upper()}' topic")
    print(f"{'='*100}\n")
    
    return view_topic(topic, limit=k)

# Available topics
TOPICS_INFO = {
    'orders': {
        'description': 'Customer Orders (Streaming)',
        'fields': ['order_id', 'user_id', 'product_id', 'amount', 'currency', 'ts'],
        'count': 'All orders placed'
    },
    'payments': {
        'description': 'Payment Events (Streaming)',
        'fields': ['payment_id', 'order_id', 'method', 'status', 'amount', 'ts'],
        'count': 'All payment transactions'
    },
    'shipments': {
        'description': 'Shipment Events (Streaming)',
        'fields': ['shipment_id', 'order_id', 'eta_days', 'carrier', 'ts'],
        'count': 'All shipments'
    },
    'delivery': {
        'description': 'Delivery Status Updates (Streaming)',
        'fields': ['delivery_id', 'shipment_id', 'status', 'reason', 'ts'],
        'count': 'DELIVERED/FAILED/RETURNED'
    },
    'users': {
        'description': 'User Changes via CDC (Debezium)',
        'fields': ['user_id', 'name', 'email', 'city', 'phone', 'op', 'ts_ms'],
        'count': 'before/after with operation'
    },
    'products': {
        'description': 'Product Changes via CDC (Debezium)',
        'fields': ['product_id', 'name', 'category', 'price', 'supplier_id', 'op', 'ts_ms'],
        'count': 'before/after with operation'
    }
}

print("\n" + "="*100)
print("📌 TOPIC SELECTOR & SAMPLE VIEWER")
print("="*100 + "\n")

print("Available topics:\n")
for i, (topic, info) in enumerate(TOPICS_INFO.items(), 1):
    print(f"{i}. {topic.upper():<12} - {info['description']}")
    print(f"   Fields: {', '.join(info['fields'][:3])}...")
    print()

print("\nUsage:")
print("  show_samples('orders', 5)       # Show 5 orders")
print("  show_samples('users', 3)        # Show 3 user changes")
print("  show_samples('payments', 10)    # Show 10 payments")
print("  show_samples('delivery', 2)     # Show 2 delivery status")
print()

print("Or call view_topic for more details:")
print("  view_topic('orders', limit=5)   # Full detailed view")



📌 TOPIC SELECTOR & SAMPLE VIEWER

Available topics:

1. ORDERS       - Customer Orders (Streaming)
   Fields: order_id, user_id, product_id...

2. PAYMENTS     - Payment Events (Streaming)
   Fields: payment_id, order_id, method...

3. SHIPMENTS    - Shipment Events (Streaming)
   Fields: shipment_id, order_id, eta_days...

4. DELIVERY     - Delivery Status Updates (Streaming)
   Fields: delivery_id, shipment_id, status...

5. USERS        - User Changes via CDC (Debezium)
   Fields: user_id, name, email...

6. PRODUCTS     - Product Changes via CDC (Debezium)
   Fields: product_id, name, category...


Usage:
  show_samples('orders', 5)       # Show 5 orders
  show_samples('users', 3)        # Show 3 user changes
  show_samples('payments', 10)    # Show 10 payments
  show_samples('delivery', 2)     # Show 2 delivery status

Or call view_topic for more details:
  view_topic('orders', limit=5)   # Full detailed view


In [57]:
# 🚀 DEMO: Using show_samples to view topics

print("\n" + "="*100)
print("🚀 DEMO: How to use show_samples()")
print("="*100 + "\n")

print("Example 1: Show 10 users (CDC changes)")
print("-" * 100)
print("Code: show_samples('users', 200)\n")

df_users_10 = show_samples('users', 200)



🚀 DEMO: How to use show_samples()

Example 1: Show 10 users (CDC changes)
----------------------------------------------------------------------------------------------------
Code: show_samples('users', 200)


📋 Showing 200 samples from 'USERS' topic


📊 TOPIC: USERS | File: 00172-99646-62a63434-e5fd-4461-915c-f0c567d961b7-76-00001.parquet
Total records: 158 | Showing: 158

────────────────────────────────────────────────────────────────────────────────────────────────────
Record #1:
────────────────────────────────────────────────────────────────────────────────────────────────────
  event_source                   = demo.public.users
  event_time                     = 2026-04-06 18:06:04.959000+00:00
  before                         = {'id': 2009, 'user_id': 'usr_agf1bgkc', 'email': 'updated_3054@example.com', 'co
  after                          = {'id': 2009, 'user_id': 'usr_agf1bgkc', 'email': 'updated_3054@example.com', 'co
  source                         = {'version': '3.0.0.Fi